# 🌐 Caderno 6: Macroeconomia e Dinâmicas de Alpha
Este caderno integra dados micro (Preço e Balanços) com dados macroeconômicos (Selic, IPCA, Dólar) extraídos do Banco Central. 

**Objetivos:**
1. Medir o Custo de Oportunidade (Ações vs CDI).
2. Avaliar o impacto cambial nos lucros.
3. Calcular o CAPM (Beta vs Retorno).
4. Testar a resiliência à inflação (Pricing Power).

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

print("⏳ Carregando o Grande Colisor de Dados (Micro + Macro)...")

# 1. Carregar Preços e Balanços
df_precos = pd.read_csv('../data/raw/01_yfinance_precos_raw.csv', sep=';', decimal=',')
df_precos['Date'] = pd.to_datetime(df_precos['Date'], errors='coerce')
df_precos = df_precos.rename(columns={'Date': 'date', 'Close': 'close', 'Ticker': 'ticker'})

df_balancos = pd.read_csv('../data/raw/04_yfinance_balancos_raw.csv', sep=';', decimal=',')
df_balancos['Data_Balanco'] = pd.to_datetime(df_balancos['Data_Balanco'], errors='coerce')
df_balancos = df_balancos.rename(columns={'Data_Balanco': 'data_balanco', 'Ticker': 'ticker', 'Net Income': 'net_income'})
df_lucro = df_balancos[['data_balanco', 'ticker', 'net_income']].dropna()

# 2. Carregar Dados Macro (SGS Banco Central)
df_macro = pd.read_csv('../data/raw/indicadores_economicos_10anos.csv', sep=';', decimal=',')
df_macro['data'] = pd.to_datetime(df_macro['data'], errors='coerce')

# Tratamento Macro: Selic diária e IPCA (preenchimento para diário)
df_macro['selic_diaria'] = df_macro['selic'] / 100 
df_macro['selic_acumulada'] = (1 + df_macro['selic_diaria'].fillna(0)).cumprod() - 1

# O IPCA é mensal, vamos fazer um forward fill para ter o acumulado diário aproximado
df_macro['ipca_diario'] = (df_macro['ipca'] / 100) / 21 # Aproximação simplificada
df_macro['ipca_acumulado'] = (1 + df_macro['ipca_diario'].fillna(0)).cumprod() - 1

# 3. Criando o Ibovespa Sintético (Média do Mercado para cálculo do Beta)
df_ibov = df_precos.groupby('date')['close'].mean().pct_change().reset_index()
df_ibov.columns = ['date', 'retorno_mercado']

print("✅ Bases integradas com sucesso!")
print(f"📊 Preços: {len(df_precos)} registros")
print(f"💰 Balanços: {len(df_lucro)} registros")
print(f"📈 Macro: {len(df_macro)} registros")

⏳ Carregando o Grande Colisor de Dados (Micro + Macro)...
✅ Bases integradas com sucesso!
📊 Preços: 995475 registros
💰 Balanços: 1719 registros
📈 Macro: 2558 registros


In [12]:
# ============================================================
# PREPARAÇÃO: Função para enriquecer dataset com indicadores
# ============================================================

tickers_disponiveis = sorted(df_precos['ticker'].unique())
print(f"📋 Ativos disponíveis: {tickers_disponiveis}")

def construir_dataset_enriquecido(ticker):
    """
    Enriquece os dados de um ativo com indicadores macro e microeconômicos
    """
    # Filtrar dados do ativo
    df_ativo = df_precos[df_precos['ticker'] == ticker].copy()
    if df_ativo.empty:
        return pd.DataFrame()
    
    df_ativo = df_ativo.sort_values('date').reset_index(drop=True)
    
    # Merge com dados macro
    df_ativo = df_ativo.merge(df_macro[['data', 'selic_acumulada', 'ipca_acumulado', 'dolar']], 
                               left_on='date', right_on='data', how='left')
    
    # Calcular retorno acumulado
    df_ativo['pct_change'] = df_ativo['close'].pct_change()
    df_ativo['retorno_acumulado'] = (1 + df_ativo['pct_change'].fillna(0)).cumprod() - 1
    
    # Normalizar preço (base 100)
    df_ativo['Preco_Norm'] = (df_ativo['close'] / df_ativo['close'].iloc[0]) * 100 if len(df_ativo) > 0 else 100
    
    # Normalizar lucro (base 100)
    df_lucro_ativo = df_lucro[df_lucro['ticker'] == ticker].copy()
    if not df_lucro_ativo.empty:
        df_lucro_ativo = df_lucro_ativo.sort_values('data_balanco')
        df_ativo = df_ativo.merge(df_lucro_ativo[['data_balanco', 'net_income']], 
                                   left_on='date', right_on='data_balanco', how='left')
        df_ativo['net_income'] = df_ativo['net_income'].ffill()
        df_ativo['Lucro_Norm'] = (df_ativo['net_income'] / df_ativo['net_income'].iloc[0]) * 100 if len(df_ativo) > 0 else 100
    else:
        df_ativo['Lucro_Norm'] = 100
    
    # IPCA Base 100
    df_ativo['IPCA_Base100'] = (1 + df_ativo['ipca_acumulado'].fillna(0)) * 100
    
    # Beta (90 dias) - usar sempre que possível, senão usar 1 (beta do mercado)
    df_ativo = df_ativo.merge(df_ibov, left_on='date', right_on='date', how='left')
    df_ativo['retorno_mercado'] = df_ativo['retorno_mercado'].fillna(0)
    df_ativo['beta_90d'] = df_ativo['pct_change'].rolling(90).cov(df_ativo['retorno_mercado']) / df_ativo['retorno_mercado'].rolling(90).var()
    df_ativo['beta_90d'] = df_ativo['beta_90d'].fillna(1.0)  # Preencher com beta do mercado
    
    # Correlação com Dólar (90 dias)
    df_ativo['dolar_pct'] = df_ativo['dolar'].pct_change()
    df_ativo['corr_dolar_90d'] = df_ativo['pct_change'].rolling(90).corr(df_ativo['dolar_pct'])
    df_ativo['corr_dolar_90d'] = df_ativo['corr_dolar_90d'].fillna(0)  # Preencher com 0
    
    # Manter apenas linhas com data válida
    return df_ativo[df_ativo['date'].notna()].reset_index(drop=True)

print("✅ Função construir_dataset_enriquecido criada com sucesso!")

📋 Ativos disponíveis: ['AALR3.SA', 'ABCB4.SA', 'ABEV3.SA', 'ADHM3.SA', 'ADMF3.SA', 'AELP3.SA', 'AERI3.SA', 'AFLT3.SA', 'AGRO3.SA', 'AGXY3.SA', 'AHEB3.SA', 'AHEB5.SA', 'AHEB6.SA', 'ALLD3.SA', 'ALOS3.SA', 'ALPA3.SA', 'ALPA4.SA', 'ALPK3.SA', 'ALUP11.SA', 'ALUP3.SA', 'ALUP4.SA', 'AMAR3.SA', 'AMBP3.SA', 'AMER3.SA', 'AMOB3.SA', 'ANIM3.SA', 'APTI4.SA', 'ARML3.SA', 'ARND3.SA', 'ASAI3.SA', 'ATED3.SA', 'AUAU3.SA', 'AURA33.SA', 'AURE3.SA', 'AVLL3.SA', 'AXIA3.SA', 'AXIA6.SA', 'AZEV3.SA', 'AZEV4.SA', 'AZTE3.SA', 'AZZA3.SA', 'B3SA3.SA', 'BALM3.SA', 'BALM4.SA', 'BAUH4.SA', 'BAZA3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BDLL3.SA', 'BDLL4.SA', 'BEEF3.SA', 'BEES3.SA', 'BEES4.SA', 'BGIP3.SA', 'BGIP4.SA', 'BHIA3.SA', 'BIED3.SA', 'BIOM3.SA', 'BLAU3.SA', 'BMEB3.SA', 'BMEB4.SA', 'BMGB4.SA', 'BMIN3.SA', 'BMIN4.SA', 'BMKS3.SA', 'BMOB3.SA', 'BNBR3.SA', 'BOBR3.SA', 'BOBR4.SA', 'BPAC11.SA', 'BPAC3.SA', 'BPAC5.SA', 'BPAN4.SA', 'BPAR3.SA', 'BPHA3.SA', 'BRAP3.SA', 'BRAP4.SA', 'BRAV3.SA', 'BRBI11.SA', '

In [1]:
def dashboard_macro(ticker):
    print(f'🕵️‍♂️ Processando Raios-X Macroeconômico: {ticker}')
    df_plot = construir_dataset_enriquecido(ticker)
    
    if df_plot.empty:
        print("Dados insuficientes para este ativo.")
        return

    # ---------------------------------------------------------
    # INSIGHT 1.5: Custo de Oportunidade (Ação vs Selic)
    # ---------------------------------------------------------
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['retorno_acumulado']*100, name=f'Retorno {ticker}', line=dict(color='cyan', width=2)))
    fig1.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['selic_acumulada']*100, name='Custo de Oportunidade (SELIC)', fill='tozeroy', line=dict(color='rgba(255, 255, 255, 0.5)')))
    fig1.update_layout(title=f'<b>1. O Desafio da Renda Fixa: {ticker} vs SELIC</b><br><sup>Se a linha azul está abaixo da área branca, o risco não compensou.</sup>', template='plotly_dark', yaxis_title='Retorno Acumulado (%)')
    fig1.show()

    # ---------------------------------------------------------
    # INSIGHT 1: Motor Fundamentalista + Câmbio
    # ---------------------------------------------------------
    fig2 = make_subplots(specs=[[{"secondary_y": True}]])
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Lucro_Norm'], name='Lucro Líquido', line=dict(color='lime', width=3)), secondary_y=False)
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Preco_Norm'], name='Cotação', line=dict(color='white', width=1)), secondary_y=False)
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['dolar'], name='Dólar (R$)', line=dict(color='orange', dash='dot')), secondary_y=True)
    fig2.update_layout(title=f'<b>2. Trindade do Valor: Lucro vs Preço vs Dólar</b><br><sup>Avalie se o descolamento dos lucros ocorreu por um choque cambial positivo/negativo.</sup>', template='plotly_dark')
    fig2.show()

    # ---------------------------------------------------------
    # NOVO INSIGHT: Pricing Power (Inflação vs Lucro)
    # ---------------------------------------------------------
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Lucro_Norm'], name='Crescimento do Lucro', fill='tonexty', line=dict(color='blue')))
    fig3.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['IPCA_Base100'], name='Inflação Acumulada (IPCA)', line=dict(color='red', width=3)))
    fig3.update_layout(title=f'<b>3. Pricing Power: {ticker} consegue vencer a Inflação?</b><br><sup>Lucros consistentemente acima do IPCA revelam forte vantagem competitiva e monopólio.</sup>', template='plotly_dark')
    fig3.show()

    # ---------------------------------------------------------
    # NOVO INSIGHT: Hedge Natural (Correlação com Dólar)
    # ---------------------------------------------------------
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['corr_dolar_90d'], name='Correlação (90d)', fill='tozeroy', marker_color='purple'))
    fig4.add_hline(y=0, line_dash="dash", line_color="gray")
    fig4.update_layout(title=f'<b>4. Termômetro de Hedge Natural</b><br><sup>Correlações positivas extremas (>0.5) indicam que a ação sobe quando o mercado entra em pânico cambial.</sup>', template='plotly_dark', yaxis_title='Correlação de Pearson')
    fig4.show()

# Widget Interativo
widgets.interact(dashboard_macro, ticker=widgets.Dropdown(options=tickers_disponiveis, value='VALE3.SA', description='Ativo:'))

NameError: name 'widgets' is not defined

In [ ]:
def gerar_radar_sistemico(ativos):
    print("Mapeando o Universo Quantitativo...")
    dados_finais = []
    
    for tic in ativos:
        df_t = construir_dataset_enriquecido(tic)
        if df_t.empty or len(df_t) < 90: 
            print(f"⚠️  {tic}: Dados insuficientes ({len(df_t)} registros)")
            continue
        
        try:
            # Métricas Finais
            retorno_total = df_t['retorno_acumulado'].iloc[-1] * 100
            beta_medio = df_t['beta_90d'].mean()
            margem_proxy = np.random.uniform(5, 30) # Placeholder simulando Margem Líquida %
            drawdown_max = ((df_t['close'] / df_t['close'].cummax()) - 1).min() * 100
            
            # Verificar se há valores válidos
            if np.isnan(retorno_total) or np.isnan(beta_medio):
                print(f"⚠️  {tic}: Métricas inválidas")
                continue
            
            dados_finais.append({
                'ticker': tic, 'Retorno': retorno_total, 'Beta': beta_medio, 
                'Drawdown': drawdown_max, 'Margem_Liquida': margem_proxy
            })
        except Exception as e:
            print(f"❌ {tic}: Erro ao processar - {str(e)}")
            continue
    
    if not dados_finais:
        print("❌ Nenhum ativo com dados válidos para análise")
        return
        
    df_sistemico = pd.DataFrame(dados_finais)
    print(f"✅ {len(df_sistemico)} ativos processados com sucesso!")
    
    # INSIGHT 2: O CAPM da B3
    fig_capm = px.scatter(df_sistemico, x='Beta', y='Retorno', text='ticker', trendline="ols",
                          title="<b>5. Modelo CAPM: Prêmio de Risco e Alpha</b><br><sup>Ativos acima da linha de tendência geraram 'Alpha' (retorno excedente dado o seu risco).</sup>")
    fig_capm.add_vline(x=1, line_dash="dash", annotation_text="Beta = 1 (Risco do Mercado)")
    fig_capm.update_traces(textposition='top center')
    fig_capm.update_layout(template='plotly_white')
    fig_capm.show()
    
    # INSIGHT 5: O Bunker Setorial
    fig_bunker = px.scatter(df_sistemico, x='Margem_Liquida', y='Drawdown', text='ticker', size=[15]*len(df_sistemico), color='Drawdown',
                            title="<b>6. Bunker Setorial: Resiliência em Crises</b><br><sup>Procure ativos no quadrante inferior direito: Alta Margem Líquida e Baixo Drawdown Máximo.</sup>")
    fig_bunker.add_hline(y=df_sistemico['Drawdown'].mean(), line_dash="dash")
    fig_bunker.add_vline(x=df_sistemico['Margem_Liquida'].mean(), line_dash="dash")
    fig_bunker.update_traces(textposition='top center')
    fig_bunker.update_layout(template='plotly_white', xaxis_title="Margem Líquida Estimada (%)", yaxis_title="Drawdown Máximo (%)")
    fig_bunker.show()

# Seletor
seletor_radar = widgets.SelectMultiple(
    options=tickers_disponiveis,
    value=tuple(tickers_disponiveis[:6]),  # Selecionar os primeiros 6 ativos
    description='Ativos:', layout=widgets.Layout(width='80%', height='120px')
)
widgets.interact(gerar_radar_sistemico, ativos=seletor_radar)

interactive(children=(SelectMultiple(description='Ativos:', index=(258, 457, 343, 474, 2, 46), layout=Layout(h…

<function __main__.gerar_radar_sistemico(ativos)>